In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install tf-keras

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
df=pd.read_csv("train.csv")

df=df.dropna()


df= df.sample(frac=0.20, random_state=42) 

df.head()
# df.shape[0]


df_t=pd.read_csv("test.csv")
df_t=df_t.dropna()


# cat_path=df['category_path']

org_qr=df['origin_query']
print(df.shape[0])
print(df_t.shape[0])
print(df['label'].sum())

In [ ]:
df.head()
# df.isna().sum()

In [ ]:
cat_path = df['category_path'].str.replace(",", " ", regex=False)

In [ ]:
cat_path.dtype

In [ ]:
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:

from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('sentence-transformers/LaBSE')

def calculate_product_score(product_name, tag_sentence, model=model):
    if not isinstance(product_name, str) or not isinstance(tag_sentence, str) or not tag_sentence.strip():
        return np.nan
    product_embedding = model.encode(product_name, convert_to_tensor=True)
    tag_embedding = model.encode(tag_sentence, convert_to_tensor=True)
    similarity = util.cos_sim(product_embedding, tag_embedding).item()
    return similarity

# Apply function row-wise
# df['Score'] = df.apply(lambda row: calculate_product_score(row['origin_query'], row['category_path']), axis=1)

In [ ]:
# df['Score'] = df.apply(lambda row: calculate_product_score(row['origin_query'], row['category_path']), axis=1)
# print(calculate_product_score("Shirt",["Clothing","Apparel","Cotton"]))
# print(calculate_product_score("Shirt",["Shirt","Apparel","Cotton"]))
df['score'] = [calculate_product_score(name, tags) for name, tags in zip(org_qr.values, cat_path.values)]

In [ ]:
df.head()

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix
df['result']=[0 if scr<0.20 else 1 for scr in df['score']]
df.head()

f1_sc=f1_score(df['label'],df['result'])

df['true'] = [1 if lbl == rslt else 0 for lbl, rslt in zip(df['label'].values, df['result'].values)]
df.head()
df['true'].sum()


# Precision
precision = precision_score(df['label'], df['result'])
print("Precision:", precision)

# Recall
recall = recall_score(df['label'], df['result'])
print("Recall:", recall)
cm = confusion_matrix(df['label'], df['result'])
print(cm)
print(f1_sc)

print(df['true'].sum())


from sklearn.metrics import classification_report

print(classification_report(df['label'],df['result']))